In [33]:
import os
import json
import shutil
import pandas as pd
from datasets import Dataset


In [34]:
# Configuration
data_raw_dir = "../../data/raw/xfacta"
data_processed_dir = "../../data/processed"

In [35]:
# Converts JSON files to JSONL
JSONL_EXTENSION = ".jsonl"
JSON_EXTENSION = ".json"


def _parse_content_to_items(content):
    """Return a list of JSON items parsed from either a JSON array or JSONL content."""
    try:
        parsed = json.loads(content)
        return parsed if isinstance(parsed, list) else [parsed]
    except json.JSONDecodeError:
        items = []
        for line in content.split('\n'):
            line = line.strip()
            if not line:
                continue
            try:
                items.append(json.loads(line))
            except json.JSONDecodeError:
                continue
        return items


def _normalize_label(item):
    label = None
    if "oooc_tweet" in item:
        oooc = item["oooc_tweet"]
        metadata = oooc.get("metadata", {})
        label = metadata.get("label")
    else:
        label = item.get("label")

    if isinstance(label, bool):
        return 1 if label else 0
    if isinstance(label, str):
        return 1 if label.lower() == "true" else 0
    try:
        return int(label)
    except Exception:
        return 0


def _process_file_to_jsonl(input_path, output_path, filename):
    try:
        with open(input_path, "r", encoding="utf-8", errors="replace") as infile:
            content = infile.read().strip()
            items = _parse_content_to_items(content)

        with open(output_path, "w", encoding="utf-8") as outfile:
            for item in items:
                try:
                    if not isinstance(item, dict):
                        continue
                    item["label"] = _normalize_label(item)
                    outfile.write(json.dumps(item) + "\n")
                except Exception as e:
                    print("Error processing item in %s: %s" % (filename, str(e)))
        return True
    except Exception as e:
        print("✗ Error converting %s: %s" % (filename, str(e)))
        return False


def convert_json_to_jsonl(source_dir):
    """Convert all .json files to .jsonl with binary labels (1=true, 0=false)"""
    converted_files = []

    for root, dirs, files in os.walk(source_dir):
        for file in files:
            if file.endswith(JSON_EXTENSION):
                input_path = os.path.join(root, file)
                output_path = input_path.replace(JSON_EXTENSION, JSONL_EXTENSION)
                ok = _process_file_to_jsonl(input_path, output_path, file)
                if ok:
                    converted_files.append(output_path)
                    print("✓ Converted: %s" % output_path)

    return converted_files


converted_files = convert_json_to_jsonl(data_raw_dir)
print("\n✓ Total files converted: %d\n" % len(converted_files))

✓ Converted: ../../data/raw/xfacta\dev.jsonl
✓ Converted: ../../data/raw/xfacta\test.jsonl
✓ Converted: ../../data/raw/xfacta\fake_sample\batch1.jsonl
✓ Converted: ../../data/raw/xfacta\fake_sample\batch10.jsonl
✓ Converted: ../../data/raw/xfacta\fake_sample\batch11.jsonl
✓ Converted: ../../data/raw/xfacta\fake_sample\batch12.jsonl
✓ Converted: ../../data/raw/xfacta\fake_sample\batch2.jsonl
✓ Converted: ../../data/raw/xfacta\fake_sample\batch3.jsonl
✓ Converted: ../../data/raw/xfacta\fake_sample\batch4.jsonl
✓ Converted: ../../data/raw/xfacta\fake_sample\batch5.jsonl
✓ Converted: ../../data/raw/xfacta\fake_sample\batch6.jsonl
✓ Converted: ../../data/raw/xfacta\fake_sample\batch7.jsonl
✓ Converted: ../../data/raw/xfacta\fake_sample\batch8.jsonl
✓ Converted: ../../data/raw/xfacta\fake_sample\batch9.jsonl
✓ Converted: ../../data/raw/xfacta\real_sample\batch1.jsonl
✓ Converted: ../../data/raw/xfacta\real_sample\batch10.jsonl
✓ Converted: ../../data/raw/xfacta\real_sample\batch11.jsonl
✓ Co

In [36]:
# Copy JSONL files to processed directory
def copy_jsonl_to_processed(source_dir, dest_dir):
    """Copy all .jsonl files to processed directory"""
    copied_files = []

    for root, dirs, files in os.walk(source_dir):
        for file in files:
            if file.endswith(JSONL_EXTENSION):
                source_path = os.path.join(root, file)
                rel_path = os.path.relpath(source_path, source_dir)
                dest_path = os.path.join(dest_dir, rel_path)

                os.makedirs(os.path.dirname(dest_path), exist_ok=True)
                shutil.copy2(source_path, dest_path)
                copied_files.append(dest_path)
                print("✓ Copied: %s" % dest_path)

    return copied_files

copied_files = copy_jsonl_to_processed(data_raw_dir, data_processed_dir)
print("\n✓ Total files copied: %d\n" % len(copied_files))

✓ Copied: ../../data/processed\dev.jsonl
✓ Copied: ../../data/processed\test.jsonl
✓ Copied: ../../data/processed\fake_sample\batch1.jsonl
✓ Copied: ../../data/processed\fake_sample\batch10.jsonl
✓ Copied: ../../data/processed\fake_sample\batch11.jsonl
✓ Copied: ../../data/processed\fake_sample\batch12.jsonl
✓ Copied: ../../data/processed\fake_sample\batch2.jsonl
✓ Copied: ../../data/processed\fake_sample\batch3.jsonl
✓ Copied: ../../data/processed\fake_sample\batch4.jsonl
✓ Copied: ../../data/processed\fake_sample\batch5.jsonl
✓ Copied: ../../data/processed\fake_sample\batch6.jsonl
✓ Copied: ../../data/processed\fake_sample\batch7.jsonl
✓ Copied: ../../data/processed\fake_sample\batch8.jsonl
✓ Copied: ../../data/processed\fake_sample\batch9.jsonl
✓ Copied: ../../data/processed\real_sample\batch1.jsonl
✓ Copied: ../../data/processed\real_sample\batch10.jsonl
✓ Copied: ../../data/processed\real_sample\batch11.jsonl
✓ Copied: ../../data/processed\real_sample\batch12.jsonl
✓ Copied: ../..

In [37]:
# Delete JSONL files from raw directory and helper to load data

def _extract_example_fields(example):
    if "oooc_tweet" in example:
        oooc = example["oooc_tweet"]
        text = oooc.get("text", "")
        images = oooc.get("images", [])
        metadata = oooc.get("metadata", {})
        label = metadata.get("label")
    else:
        text = example.get("text", "")
        images = example.get("images", [])
        label = example.get("label")

    if isinstance(label, bool):
        label = 1 if label else 0
    elif isinstance(label, str):
        label = 1 if label.lower() == "true" else 0

    return {"text": text, "images": images, "label": label}


def delete_jsonl_from_raw(source_dir):
    """Delete all .jsonl files from raw data directory"""
    deleted_files = []

    for root, dirs, files in os.walk(source_dir):
        for file in files:
            if file.endswith(JSONL_EXTENSION):
                file_path = os.path.join(root, file)
                os.remove(file_path)
                deleted_files.append(file_path)
                print("✓ Deleted: %s" % file_path)

    return deleted_files


def load_processed_data(source_dir):
    """Load all JSONL files into a Hugging Face Dataset"""
    all_data = []

    for root, dirs, files in os.walk(source_dir):
        for file in files:
            if file.endswith(JSONL_EXTENSION):
                file_path = os.path.join(root, file)
                print("Loading %s..." % file)
                with open(file_path, "r", encoding="utf-8") as f:
                    for line in f:
                        try:
                            example = json.loads(line)
                            item = _extract_example_fields(example)
                            all_data.append(item)
                        except Exception as e:
                            print("Error parsing %s: %s" % (file, str(e)))

    return Dataset.from_pandas(pd.DataFrame(all_data))


deleted_files = delete_jsonl_from_raw(data_raw_dir)
print("\n✓ Total files deleted: %d\n" % len(deleted_files))

✓ Deleted: ../../data/raw/xfacta\dev.jsonl
✓ Deleted: ../../data/raw/xfacta\test.jsonl
✓ Deleted: ../../data/raw/xfacta\fake_sample\batch1.jsonl
✓ Deleted: ../../data/raw/xfacta\fake_sample\batch10.jsonl
✓ Deleted: ../../data/raw/xfacta\fake_sample\batch11.jsonl
✓ Deleted: ../../data/raw/xfacta\fake_sample\batch12.jsonl
✓ Deleted: ../../data/raw/xfacta\fake_sample\batch2.jsonl
✓ Deleted: ../../data/raw/xfacta\fake_sample\batch3.jsonl
✓ Deleted: ../../data/raw/xfacta\fake_sample\batch4.jsonl
✓ Deleted: ../../data/raw/xfacta\fake_sample\batch5.jsonl
✓ Deleted: ../../data/raw/xfacta\fake_sample\batch6.jsonl
✓ Deleted: ../../data/raw/xfacta\fake_sample\batch7.jsonl
✓ Deleted: ../../data/raw/xfacta\fake_sample\batch8.jsonl
✓ Deleted: ../../data/raw/xfacta\fake_sample\batch9.jsonl
✓ Deleted: ../../data/raw/xfacta\real_sample\batch1.jsonl
✓ Deleted: ../../data/raw/xfacta\real_sample\batch10.jsonl
✓ Deleted: ../../data/raw/xfacta\real_sample\batch11.jsonl
✓ Deleted: ../../data/raw/xfacta\real_

In [ ]:
# Load dataset
dataset = load_processed_data(data_processed_dir)
print("\n✓ Dataset loaded successfully")
print("  - Total samples: %d" % len(dataset))
print("  - Features: %s\n" % dataset.column_names)

# Save dataset to disk for training
dataset.save_to_disk(os.path.join(data_processed_dir, "hf_dataset"))
print("✓ Dataset saved to %s/hf_dataset" % data_processed_dir)

Loading dev.jsonl...
Loading test.jsonl...
Loading batch1.jsonl...
Loading batch10.jsonl...
Loading batch11.jsonl...
Loading batch12.jsonl...
Loading batch2.jsonl...
Loading batch3.jsonl...
Loading batch4.jsonl...
Loading batch5.jsonl...
Loading batch6.jsonl...
Loading batch7.jsonl...
Loading batch8.jsonl...
Loading batch9.jsonl...
Loading batch1.jsonl...
Loading batch10.jsonl...
Loading batch11.jsonl...
Loading batch12.jsonl...
Loading batch2.jsonl...
Loading batch3.jsonl...
Loading batch4.jsonl...
Loading batch5.jsonl...
Loading batch6.jsonl...
Loading batch7.jsonl...
Loading batch8.jsonl...
Loading batch9.jsonl...

✓ Dataset loaded successfully
  - Total samples: 4800
  - Features: ['text', 'images', 'label']



Saving the dataset (1/1 shards): 100%|██████████| 4800/4800 [00:00<00:00, 1149714.99 examples/s]

✓ Dataset saved to ../../data/processed/hf_dataset


: 